<h1><center><strong></strong></center></h1>
<h1><center><strong>CME466</strong></center></h1>
<h1><center><strong>Design of an Advanced Digital System</strong></center></h1>
<p><center><strong>Department of Electrical and Computer Engineering</strong></center></p>
<p><center><strong>2026 Winter Term</strong></center></p>
<p><center><strong>Last update: March 11,2026</strong></center></p>

# CNN Part 3 - Transfer Learning
Food-101 dataset, EfficientNet



# 0. Prerequisites

## 0.1 Import Libraries

In [ ]:
# Import PyTorch
import torch
from torch import nn

# Import torchvision
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor

# Import matplotlib for visualization
import matplotlib.pyplot as plt
from pathlib import Path

# Check versions
# print(f"PyTorch version: {torch.__version__}\ntorchvision version: {torchvision.__version__}")

In [ ]:
# Try to import torchinfo, install if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] coudln't find torchinfo, installing it...")
    !pip install -q torchinfo
    from torchinfo import summary
    print("[INFO] Done!")

In [ ]:
# Try to import torchmetrics and install if it doesn't work
try:
    import torchmetrics
except:
    print("[INFO] coudln't find torchmetrics, installing it...")
    !pip install -q torchmetrics
    import torchmetrics
    print("[INFO] Done!")

## 0.2 Setup Device agnostic code

In [ ]:
# Set up device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
%matplotlib inline

# 1. Getting a dataset

## 1.1 Custom Dataset - Food-101

<p>The data we're going to be using is a subset of the <a href="https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/">Food-101 dataset</a>.</p>
<p>Food101 is popular computer vision benchmark as it contains 1000 images of 101 different kinds of foods (i.e., 101 classes), totaling 101,000 images (75,750 train and 25,250 test).</p>

![Food101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/static/img/food-101.jpg)


## 1.2 Download the dataset

### Download the dataset directly into colab
We are using a subset of Food-101 dataset that contains only 5 classes.

In [ ]:
!gdown --id 1ARrYMBASqsPomwD8RsgwyPxzqU09igi6 -O food101_5_cls.zip

### Extracting the dataset

In [ ]:
import requests
import zipfile
from pathlib import Path

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "food101_5_cls"

# If the image folder doesn't exist, create it...
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

# Unzip the dataset
with zipfile.ZipFile("food101_5_cls.zip", "r") as zip_ref:
    print("Unzipping the dataset ...")
    zip_ref.extractall(image_path)

<p>We can inspect what's in our data directory by writing a small helper function to walk through each of the subdirectories and count the files present.</p>
<p>To do so, we'll use Python's in-built <a href="https://docs.python.org/3/library/os.html#os.walk"><code>os.walk()</code></a>.</p>
We can also check the folders on the left.....

In [ ]:
import os
def walk_through_dir(dir_path):
  """
  Walks through dir_path returning its contents.
  Args:
    dir_path (str or pathlib.Path): target directory

  Returns:
    A print out of:
      number of subdiretories in dir_path
      number of images (files) in each subdirectory
      name of each subdirectory
  """
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

In [ ]:
walk_through_dir(image_path)

## 1.3 Display an image

### Setup our training and testing paths

In [ ]:
# Setup train and testing paths
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

In [ ]:
import random
from PIL import Image

# 1. Get all image paths (* means "any combination")
image_path_list = list(image_path.glob("*/*/*.jpg"))

# 2. Get random image path
random_image_path = random.choice(image_path_list)

# 3. Get image class from path name (the image class is the name of the directory where the image is stored)
image_class = random_image_path.parent.stem

# 4. Open image
img = Image.open(random_image_path)

# 5. Print metadata
print(f"Random image path: {random_image_path}")
print(f"Image class: {image_class}")
print(f"Image height: {img.height}")
print(f"Image width: {img.width}")
img

Note that the images are of different dimensions (512 x512. or 512 x 384, so on...)

# 3. Helper Functions

In [ ]:
from torchmetrics.classification import MulticlassAccuracy

In [ ]:
#torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               num_classes,
               device: torch.device = device):
    """Evaluates a given model on a given dataset.

    Args:
        model (torch.nn.Module): A PyTorch model capable of making predictions on data_loader.
        data_loader (torch.utils.data.DataLoader): The target dataset to predict on.
        loss_fn (torch.nn.Module): The loss function of model.
        num_classes (int): Number of classes in dataset.
        device (str, optional): Target device to compute on. Defaults to device.

    Returns:
        (dict): Results of model making predictions on data_loader.
    """
    acc = MulticlassAccuracy(num_classes=num_classes, average='macro').to(device)
    loss = 0
    model.eval()
    with torch.inference_mode():
        for X, y in data_loader:
            # Send data to the target device
            X, y = X.to(device), y.to(device)
            y_logits = model(X)
            loss += loss_fn(y_logits, y)
            y_pred_probs = torch.softmax(y_logits, dim=1)
            y_pred_labels = torch.argmax(y_pred_probs, dim=1)
            acc.update(y_pred_labels, y)

        # Scale loss and acc
        loss /= len(data_loader)
        test_acc = acc.compute().cpu().numpy()
        acc.reset()

    return {"model_name": model.__class__.__name__, # only works when model was created with a class
            "model_loss": loss.item(),
            "model_acc": test_acc.item()}

In [ ]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               num_classes,
               device: torch.device = device):
    # Instantiate Multiclass Accuracy instance
    acc = MulticlassAccuracy(num_classes=num_classes, average='micro').to(device)
    train_loss = 0
    model.to(device)
    for batch, (X, y) in enumerate(data_loader):
        # Send data to GPU
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_logits = model(X)

        # 2. Calculate loss
        loss = loss_fn(y_logits, y)
        train_loss += loss
        y_pred_prob = torch.softmax(y_logits, dim=1)
        y_pred_labels = torch.argmax(y_pred_prob, dim=1)
        acc.update(y_pred_labels, y)

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

    # Calculate loss and accuracy per epoch and print out what's happening
    train_loss /= len(data_loader)
    train_acc = acc.compute().cpu().numpy()

    acc.reset()

    print(f"Train loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")

    return train_loss, train_acc

In [ ]:
def test_step(data_loader: torch.utils.data.DataLoader,
              model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              num_classes,
              device: torch.device = device):

    acc = MulticlassAccuracy(num_classes=num_classes, average='micro').to(device)
    test_loss = 0
    model.to(device)
    model.eval() # put model in eval mode
    # Turn on inference context manager
    with torch.inference_mode():
        for X, y in data_loader:
            # Send data to GPU
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_logits = model(X)

            # 2. Calculate loss and accuracy
            test_loss += loss_fn(test_logits, y)
            test_pred_probs = torch.softmax(test_logits, dim=1)
            test_pred_labels = torch.argmax(test_pred_probs, dim=1)
            acc.update(test_pred_labels, y)

        # Adjust metrics and print out
        test_loss /= len(data_loader)
        test_acc = acc.compute().cpu().numpy()

        acc.reset()
        print(f"Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%\n")
        return test_loss, test_acc

In [ ]:
import matplotlib.pyplot as plt
def visualize_training(results):

    # Plot loss
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(range(1, len(results["train_loss"]) + 1), results["train_loss"], 'bo-', label='Train Loss')
    plt.plot(range(1, len(results["train_loss"]) + 1), results["test_loss"], 'ro-', label='Test Loss')
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Training & Testing Loss")
    plt.legend()
    plt.grid()
    plt.ylim(0, max(max(results["train_loss"]), max(results["test_loss"])) * 1.1)  # Ensure the loss starts from 0

    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(range(1, len(results["train_loss"]) + 1), results["train_acc"], 'bo-', label='Train Accuracy')
    plt.plot(range(1, len(results["train_loss"]) + 1), results["test_acc"], 'ro-', label='Test Accuracy')
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy (%)")
    plt.title("Training & Testing Accuracy")
    plt.legend()
    plt.grid()
    plt.ylim(0, 1)  # Accuracy should be between 0% and 100%

    plt.show()

In [ ]:
# Plot Predictions Images
import numpy as np

def plot_image(i, predictions_array, true_label, img):
    true_label, img = true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])

    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

    predicted_label = torch.argmax(predictions_array).item()

    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'

    plt.xlabel("{} {:2.0f}% ({})".format(class_names[predicted_label],
                                100*torch.max(predictions_array).item(),
                                class_names[true_label]),
                                color=color)

def plot_value_array(i, predictions_array, true_label, num_classes=10):
  true_label = true_label[i]
  plt.grid(False)
  plt.xticks(range(num_classes))
  plt.yticks([])
  thisplot = plt.bar(range(num_classes), predictions_array, color="#777777")
  plt.ylim([0, 1])
  predicted_label = np.argmax(predictions_array)

  thisplot[predicted_label].set_color('red')
  thisplot[true_label].set_color('blue')

# Transfer Learning

**Transfer learning** is a machine learning technique where a model trained on one task is repurposed as the foundation for a second task. This approach is beneficial when the second task is related to the first or when data for the second task is limited.

## 6.1 Where to find pretrained models

<table>
<thead>
<tr>
<th><strong>Location</strong></th>
<th><strong>What's there?</strong></th>
<th><strong>Link(s)</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td><strong>PyTorch domain libraries</strong></td>
<td>Each of the PyTorch domain libraries (<code>torchvision</code>, <code>torchtext</code>) come with pretrained models of some form. The models there work right within PyTorch.</td>
<td><a href="https://pytorch.org/vision/stable/models.html"><code>torchvision.models</code></a>, <a href="https://pytorch.org/text/main/models.html"><code>torchtext.models</code></a>, <a href="https://pytorch.org/audio/stable/models.html"><code>torchaudio.models</code></a>, <a href="https://pytorch.org/torchrec/torchrec.models.html"><code>torchrec.models</code></a></td>
</tr>
<tr>
<td><strong>HuggingFace Hub</strong></td>
<td>A series of pretrained models on many different domains (vision, text, audio and more) from organizations around the world. There's plenty of different datasets too.</td>
<td><a href="https://huggingface.co/models">https://huggingface.co/models</a>, <a href="https://huggingface.co/datasets">https://huggingface.co/datasets</a></td>
</tr>
<tr>
<td><strong><code>timm</code> (PyTorch Image Models) library</strong></td>
<td>Almost all of the latest and greatest computer vision models in PyTorch code as well as plenty of other helpful computer vision features.</td>
<td><a href="https://github.com/rwightman/pytorch-image-models">https://github.com/rwightman/pytorch-image-models</a></td>
</tr>
<tr>
<td><strong>Paperswithcode</strong></td>
<td>A collection of the latest state-of-the-art machine learning papers with code implementations attached. You can also find benchmarks here of model performance on different tasks.</td>
<td><a href="https://paperswithcode.com/">https://paperswithcode.com/</a></td>
</tr>
</tbody>
</table>

## Getting a pretrained model - EfficientNet

<p>The pretrained model we're going to be using is <a href="https://pytorch.org/vision/main/models/generated/torchvision.models.efficientnet_b0.html"><code>torchvision.models.efficientnet_b0()</code></a>.</p>
<p>The architecture is from the paper <em><a href="https://arxiv.org/abs/1905.11946">EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks</a></em>.</p>

![transfer learning](https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/06-effnet-b0-feature-extractor.png)


# EfficientNet
EfficientNet is a family of highly accurate, efficient convolutional neural network (CNN) models developed by
Google AI in 2019 that achieve state-of-the-art accuracy with fewer parameters and smaller computational costs. It uses a novel "compound scaling" method to uniformly scale network depth, width, and resolution with a fixed set of scaling coefficients, rather than scaling them arbitrarily.


The family consists of 8 models ranging from B0 (smallest, fastest) to B7 (largest, highest accuracy). EfficientNet-B7 reached 84.3% top-1 accuracy on ImageNet.



## 6.2 Creating Dataloaders

When using a pretrained model, it's important that **your custom data going into the model is prepared in the same way as the original training data that went into the model.**

The pretrained model was trained using ImageNet dataset that has 1,000 classes.

In [ ]:
# NEW: Setup the model with pretrained weights and send it to the target device (torchvision v0.13+)
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # .DEFAULT = best available weights
model_2 = torchvision.models.efficientnet_b0(weights=weights).to(device)
weights

<p>And now to access the transforms associated with our <code>weights</code>, we can use the <code>transforms()</code> method.</p>
<p>This is essentially saying "get the data transforms that were used to train the <code>EfficientNet_B0_Weights</code> on ImageNet".</p>

In [ ]:
# Get the transform operations used to create the pretrained weights
import torchvision.transforms as transforms

transforms = weights.transforms()
transforms

In [ ]:
train_data = datasets.ImageFolder(root=train_dir, # target folder of images
                                  transform=transforms, # transforms to perform on data (images)
                                  target_transform=None) # transforms to perform on labels (if necessary)

test_data = datasets.ImageFolder(root=test_dir,
                                 transform=transforms)

print(f"Train data:\n{train_data}\nTest data:\n{test_data}")

In [ ]:
# Verify the lengths (same as the number of datapoints above)
len(train_data), len(test_data)

In [ ]:
# Get class names as a list
class_names = train_data.classes
class_names

In [ ]:
img, label = train_data[125][0], train_data[125][1]
print(f"Image tensor:\n{img}")
print(f"Image shape: {img.shape}")
print(f"Image datatype: {img.dtype}")
print(f"Image label: {label}")
print(f"Label datatype: {type(label)}")

In [ ]:
# Turn train and test Datasets into DataLoaders
from torch.utils.data import DataLoader

# Get class names
class_names = train_data.classes

train_dataloader = DataLoader(
      train_data,
      batch_size=32,
      shuffle=True,
      num_workers=1,
      pin_memory=True,
  )
test_dataloader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False,
    num_workers=1,
    pin_memory=True,
)

In [ ]:
img, label = next(iter(train_dataloader))

print(f"Image shape: {img.shape}") #[batch_size, color_channels, height, width]
print(f"Label shape: {label.shape}")

In [ ]:
# see some sample images
import matplotlib.pyplot as plt

def plot_transformed_images(image_paths, transform, n=3, seed=40):
    """Plots a series of random images from image_paths.

    Will open n image paths from image_paths, transform them
    with transform and plot them side by side.

    Args:
        image_paths (list): List of target image paths.
        transform (PyTorch Transforms): Transforms to apply to images.
        n (int, optional): Number of images to plot. Defaults to 3.
        seed (int, optional): Random seed for the random generator. Defaults to 42.
    """
    random.seed(seed)
    random_image_paths = random.sample(image_paths, k=n)
    for image_path in random_image_paths:
        with Image.open(image_path) as f:
            fig, ax = plt.subplots(1, 2)
            ax[0].imshow(f)
            ax[0].set_title(f"Original \nSize: {f.size}")
            ax[0].axis("off")

            # Transform and plot image
            # Note: permute() will change shape of image to suit matplotlib
            # (PyTorch default is [C, H, W] but Matplotlib is [H, W, C])
            transformed_image = transform(f).permute(1, 2, 0)
            ax[1].imshow(transformed_image)
            ax[1].set_title(f"Transformed \nSize: {transformed_image.shape}")
            ax[1].axis("off")

            fig.suptitle(f"Class: {image_path.parent.stem}", fontsize=16)

plot_transformed_images(image_path_list,
                        transform=transforms,
                        n=3)

In [ ]:
from torchinfo import summary
summary(model=model_2,
        input_size=(32, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        row_settings=["var_names"])

![efficient net b0](https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/06-v2-effnetb0-model-print-out.png)

<p><code>Efficientnet_b0</code> comes in three main parts:</p>
<ol>
<li><code>features</code> - A collection of convolutional layers and other various activation layers to learn a base representation of vision data (this base representation/collection of layers is often referred to as <strong>features</strong> or <strong>feature extractor</strong>, "the base layers of the model learn the different <strong>features</strong> of images").</li>
<li><code>avgpool</code> - Takes the average of the output of the <code>features</code> layer(s) and turns it into a <strong>feature vector</strong>.</li>
<li><code>classifier</code> - Turns the <strong>feature vector</strong> into a vector with the same dimensionality as the number of required output classes (since <code>efficientnet_b0</code> is pretrained on ImageNet and because ImageNet has 1000 classes, <code>out_features=1000</code> is the default).</li>
</ol>


Two important observations (from the above structure layer map):
1. The parameters in the CNN layers are organized as "features" (see at the beginning)
2. The final classification layer is called "classifier" (see at the end)

## 6.4 Freezing the base model and changing the output layer to suit our needs

<p>The process of transfer learning usually goes: freeze some base layers of a pretrained model (typically the <code>features</code> section) and then adjust the output layers (also called head/classifier layers) to suit your needs.</p>
<img alt="changing the efficientnet classifier head to a custom number of outputs" src="https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/06-v2-effnet-changing-the-classifier-head.png" width="900/"/>

<p><em>You can customise the outputs of a pretrained model by changing the output layer(s) to suit your problem. The original <code>torchvision.models.efficientnet_b0()</code> comes with <code>out_features=1000</code> because there are 1000 classes in ImageNet, the dataset it was trained on. However, for our problem, classifying images of pizza, steak, sushi, hot_dog, and tacos, we only need <code>out_features=5</code>.</em></p>


<blockquote>
<p><strong>Note:</strong> To <em>freeze</em> layers means to keep them how they are during training. For example, if your model has pretrained layers, to <em>freeze</em> them would be to say, "don't change any of the patterns in these layers during training, keep them how they are." In essence, we'd like to keep the pretrained weights/patterns our model has learned from ImageNet as a backbone and then only change the output layers.</p>
</blockquote>


<p>We can freeze all of the layers/parameters in the <code>features</code> section by setting the attribute <code>requires_grad=False</code>.</p>
<p>For parameters with <code>requires_grad=False</code>, PyTorch doesn't track gradient updates and in turn, these parameters won't be changed by our optimizer during training.</p>
<p>In essence, a parameter with <code>requires_grad=False</code> is "untrainable" or "frozen" in place.</p>

In [ ]:
# Freeze all base layers in the "features" section of the model (the feature extractor)
for param in model_2.features.parameters():
    param.requires_grad = False

<p>Let's now adjust the output layer or the <code>classifier</code> portion of our pretrained model to our needs.</p>
<p>We can change the <code>classifier</code> portion of our model by creating a new series of layers.</p>
<p>The current <code>classifier</code> consists of:</p>
<pre><code>(classifier): Sequential(
    (0): Dropout(p=0.2, inplace=True)
    (1): Linear(in_features=1280, out_features=1000, bias=True)
    )
</code></pre>
<p>We'll keep the <code>Dropout</code> layer the same using <a href="https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html"><code>torch.nn.Dropout(p=0.2, inplace=True)</code></a>.</p>


<blockquote>
<p><strong>Note:</strong> <a href="https://developers.google.com/machine-learning/glossary#dropout_regularization">Dropout layers</a> randomly remove connections between two neural network layers with a probability of <code>p</code>. For example, if <code>p=0.2</code>, 20% of connections between neural network layers will be removed at random each pass. This practice is meant to help regularize (prevent overfitting) a model by making sure the connections that remain learn features to compensate for the removal of the other connections (hopefully these remaining features are <em>more general</em>).</p>
</blockquote>

In [ ]:
# Another option is to freeze all layers, and then only enable training of the classifier layer
# this technique is useful for other CNN models who do not use "classifier" and "features" labels...
for param in model_2.parameters():
    param.requires_grad = False

model_2.classifier.requires_grad = True # Only train the new classification layer

In [ ]:
#torch.manual_seed(42)
# torch.cuda.manual_seed(42)

# Get the length of class_names (one output unit for each class)
# output_shape = len(class_names)

# Recreate the entire classifier layer and seed it to the target device
# model_2.classifier = torch.nn.Sequential(
#    torch.nn.Dropout(p=0.2, inplace=True),
#    torch.nn.Linear(1280, 5, bias=True)).to(device) # same as model_2.classifier#

# Another option: Just replace the last layer (which is at index -1 of the sequential block)
# to output for 5 classes
model_2.classifier[1] = nn.Linear(1280, 5).to(device)

In [ ]:
# # Do a summary *after* freezing the features and changing the output classifier layer (uncomment for actual output)
summary(model_2,
        input_size=(32, 3, 224, 224), # make sure this is "input_size", not "input_shape" (batch_size, color_channels, height, width)
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

Check from above layer map to make sure the change in output classes from 1000 to 5, and whether it is trainable.

The number of trainable parameters has also gone down to 6,405 only. This helps us quick train the model for the Food-101 dataset.

## 6.4 Train the model

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_2.parameters(),
                            lr=0.001, weight_decay=0.01)

In [ ]:
# Import tqdm for progress bar
from tqdm.auto import tqdm
# torch.manual_seed(42)

epochs = 5 # very slow, start with smaller number
# 10 or higher will not improve the performnace any further...

# Create empty results dictionary
results = {"train_loss": [],
           "train_acc": [],
           "test_loss": [],
           "test_acc": []
}

for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n---------")
    train_loss, train_acc = train_step(data_loader=train_dataloader,
                                       model=model_2,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer,
                                       num_classes=len(class_names)
                                    )
    test_loss, test_acc = test_step(data_loader=test_dataloader,
                                    model=model_2,
                                    loss_fn=loss_fn,
                                    num_classes=len(class_names)
                                )

    # Update results dictionary
    results["train_loss"].append(train_loss.detach().cpu().numpy())
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss.detach().cpu().numpy())
    results["test_acc"].append(test_acc)

In [ ]:
# Save the model
torch.save(model_2.state_dict(), 'mycnn.pt')

# check the size of the file

In [ ]:
# Plot the traning results
visualize_training(results)

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406], device=device)
std = torch.tensor([0.229, 0.224, 0.225], device=device)

# Function to reverse normalization
def unnormalize(tensor, mean, std):
    mean = mean.view(1, 3, 1, 1)  # Reshape for broadcasting
    std = std.view(1, 3, 1, 1)
    return tensor * std + mean  # Reverse normalization

In [ ]:
import random

# Plot the first X test images, their predicted labels, and the true labels.
# Color correct predictions in blue and incorrect predictions in red.
num_rows = 4
num_cols = 2
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))

# Convert dataloader into a list of batches
all_batches = list(test_dataloader)

# Pick a random batch
X_batch, y_batch = random.choice(all_batches)
X_batch, y_batch = X_batch.to(device), y_batch.to(device)

# Get model predictions
model_2.eval()  # Ensure model is in eval mode
with torch.no_grad():
    y_pred = model_2(X_batch)  # Get raw logits
    y_pred_prob = torch.softmax(y_pred, dim=1)  # Convert logits to probabilities
    y_pred_labels = torch.argmax(y_pred_prob, dim=1)  # Get predicted labels

X_batch = unnormalize(X_batch, mean, std)
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, y_pred_prob[i].cpu(), y_batch.cpu(), X_batch.squeeze().cpu())
    plt.subplot(num_rows, 2*num_cols, 2*i+2)
    plot_value_array(i, y_pred_prob[i].cpu(), y_batch.cpu(), num_classes=len(class_names))
plt.tight_layout()
plt.show()

# Plot Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from torchmetrics import ConfusionMatrix
import seaborn as sn
import pandas as pd

# Assuming 'preds' and 'target' are your accumulated tensors of predictions and true labels
# preds should be an integer tensor of predicted labels (e.g., from torch.argmax(outputs, dim=1))
# target should be an integer tensor of true labels

num_classes = 5 # Set the number of classes
metric = ConfusionMatrix(task="multiclass", num_classes=num_classes)

model_2.eval() # Set model to evaluation mode
all_preds = []
all_targets = []
with torch.no_grad():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)
        pred_logits = model_2(X)
        pred_labels = pred_logits.argmax(dim=1)
        all_preds.append(pred_labels)
        all_targets.append(y)

# Concatenate all predictions and targets
y_pred = torch.cat(all_preds)
y_true = torch.cat(all_targets)

confmat = ConfusionMatrix(num_classes=num_classes, task='multiclass')
# Ensure tensors are on CPU for torchmetrics if device is CUDA
cm_tensor = confmat(y_pred.cpu(), y_true.cpu())

# Convert to pandas DataFrame for better visualization with seaborn
df_cm = pd.DataFrame(cm_tensor.numpy(), index=range(num_classes), columns=range(num_classes))
plt.figure(figsize = (7,5))
# Use seaborn's heatmap for a nice plot
sn.heatmap(df_cm, annot=True, fmt="d", cmap='Blues')
plt.xlabel("Predicted labels")
plt.ylabel("True labels")
plt.show()

In [ ]:
# Get class names as a list
class_names = train_data.classes
class_names

# MobileNet Architecture

MobileNet is a family of lightweight, efficient convolutional neural network (CNN) architectures developed by Google for mobile and edge computing. It uses depthwise separable convolutions—breaking standard convolutions into depthwise and pointwise layers—to significantly reduce parameters and computation, enabling high-performance vision tasks (classification, detection) on resource-constrained devices.

Let us fine tune it for the Food dataset...

In [ ]:
import torch
from torchvision import models

# NEW: Setup the model with pretrained weights and send it to the target device (torchvision v0.13+)
weights = torchvision.models.MobileNet_V2_Weights.DEFAULT # .DEFAULT = best available weights
model_2 = torchvision.models.mobilenet_v2(weights=weights).to(device)
weights

# EfficientNet vs MobileNet
At this point, compare the parameters and the training time between the two models. Which one is better  (i.e., smaller in size, quicker to train, accuracy, etc.)?

# 8. Resources

<li><a href="https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/">Food101</a></li>
<li><a href="https://www.learnpytorch.io/">Learn PyTorch for Deep Learning: Zero to Mastery book</a></li>
<li><a href="https://lightning.ai/docs/torchmetrics/stable/classification/accuracy.html">Accuracy</a></li>
<li><a href="https://pytorch.org/vision/main/generated/torchvision.datasets.CIFAR10.html">CIFAR10</a></li>
<li><a href="https://www.geeksforgeeks.org/ml-introduction-to-transfer-learning/">What is Transfer Learning?</a></li>